# 🚀 Cuaderno 03 — EDO: métodos de un paso

**Métodos Numéricos · Unidad 5: Derivadas, integrales y EDO**

Temas **6.1 Fundamentos de EDO** y **6.2 Métodos de un paso** (Euler, Euler
mejorado/Heun, Runge-Kutta) — capítulo 25 del libro.

Aprendes a resolver un problema de valor inicial con
`scipy.integrate.solve_ivp` (equivalente MATLAB `ode45`), entendiendo qué
método hay detrás de cada `method=` y cuándo usarlo.

> Camino: **teoría → laboratorio → entradas/salidas → preguntas → caso →
> envío**.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
import os, sys

CUADERNO = 'u5_03_edo_un_paso'
TAREA_ID = 'U5_03_EDO_Un_Paso'
REPO_NOMBRE = 'U5 derivadas integrales y EDO'

def _raiz_local():
    b = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(b, 'lib', 'robomat.py')):
            return b
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

if ES_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = os.path.join('/content/drive/MyDrive', 'CursoMN', REPO_NOMBRE)
else:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError('Abre el cuaderno dentro de ' + REPO_NOMBRE)

sys.path.insert(0, os.path.join(REPO, 'lib'))
print('Modo       :', 'Colab' if ES_COLAB else 'local')
print('Repositorio:', REPO)

In [ ]:
#@title 📦 Librerías
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

exec(open(os.path.join(REPO, 'lib', 'robomat.py'), encoding='utf-8').read(), globals())

## 1. Teoría: fundamentos de EDO

Una **ecuación diferencial ordinaria (EDO)** relaciona una función
desconocida $y(x)$ con sus derivadas. Si solo tiene una variable
independiente es **EDO**; con dos o más, **EDP**. El **orden** es la mayor
derivada presente.

$$\frac{dy}{dx} = f(x, y), \qquad y(x_0) = y_0 \tag{PV I}$$

Un **problema de valor inicial (PVI)** da la condición $y(x_0)=y_0$ y pide la
función $y(x)$. Ejemplo canónico: el paracaidista

$$\frac{dv}{dt} = g - \frac{c}{m}v, \qquad v(0)=0 \tag{PT7.1}$$

### Métodos de un paso

Un método de **un paso** calcula $y_{i+1}$ usando solo $y_i$ y el paso $h$:

$$y_{i+1} = y_i + \phi\, h \tag{25.1}$$

- **Euler** (orden 1):  $y_{i+1} = y_i + f(x_i, y_i)\,h$.
- **Euler mejorado / Heun** (orden 2): predictor-corrector,
  $y_{i+1} = y_i + \frac{f(x_i,y_i)+f(x_{i+1},y_{i+1}^0)}{2}h$ (25.15–25.16).
- **Runge-Kutta de 4.º orden (RK4)**: pondera 4 pendientes,
  $y_{i+1} = y_i + \frac{1}{6}(k_1 + 2k_2 + 2k_3 + k_4)h$ (25.40).

> 📖 **Figuras:** `fig_25_3` (Euler vs. exacta), `fig_25_9.pdf` (Heun
> predictor/corrector), `fig_25_15.pdf` (pendientes de RK4).

**Referencia por dentro:** `lib/edo.py` tiene `euler`, `euler_mejorado` y
`runge_kutta_4`. Tú usarás `solve_ivp`, pero verlas ayuda a entender qué
pasa dentro.

## 2. Laboratorio: resolver con `solve_ivp`

Resuelve $y' = 4e^{0.8x} - 0.5y$, $y(0)=2$ (ejemplos 25.5–25.7 del libro). La
solución exacta es $y = \frac{4}{1.3}(e^{0.8x} - e^{-0.5x}) + 2e^{-0.5x}$.

In [ ]:
def f(x, y):
    return 4*np.exp(0.8*x) - 0.5*y

def exacta(x):
    return (4/1.3)*(np.exp(0.8*x) - np.exp(-0.5*x)) + 2*np.exp(-0.5*x)

# solve_ivp con RK45 (Dormand-Prince, orden 4/5) — el 'ode45' de Python
sol = solve_ivp(f, (0, 4), [2.0], method='RK45', dense_output=True)

x = np.linspace(0, 4, 50)
y_num = sol.sol(x)[0]
y_exa = exacta(x)

print('   x   |  y (RK45)  | y exacta  | error rel %')
for xi, yn, ye in [(0.5, sol.sol(0.5)[0], exacta(0.5)),
                   (1.0, sol.sol(1.0)[0], exacta(1.0)),
                   (4.0, sol.sol(4.0)[0], exacta(4.0))]:
    print(f'{xi:5.1f} | {yn:10.6f} | {ye:9.6f} | {abs(yn-ye)/ye*100:11.4f}')

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(x, y_exa, 'k-', label='exacta')
plt.plot(x, y_num, 'ro', markersize=3, label='solve_ivp (RK45)')
plt.xlabel('x'); plt.ylabel('y')
plt.legend(); plt.grid(alpha=0.3); plt.title("y' = 4e^{0.8x} - 0.5y")
plt.show()

## 3. Entradas, salidas y cuándo usar cada método

### `solve_ivp(fun, t_span, y0, method=..., t_eval=None, ...)`

- **Entradas:**
  - `fun(x, y)`: la EDO $f(x,y)$ (debe aceptar `(x, y)` y devolver un arreglo).
  - `t_span`: `(x0, xf)`, el intervalo de integración.
  - `y0`: condición inicial (arreglo).
  - `method`: `'RK45'` (RK 4/5, por omisión), `'RK23'` (RK 2/3, tipo Heun),
    `'DOP853'` (RK 8), `'LSODA'`/`'BDF'` (multipaso/rígidas).
  - `t_eval`: puntos donde quieres la salida (opcional).
- **Salidas:** objeto con `.t` (puntos), `.y` (valores) y `.sol` (interpolante).
- **Cuándo:** es el solucionador general; `RK45` cubre la mayoría de casos no
  rígidos. Usa `method='BDF'`/`'LSODA'` si el problema es **rígido**.

> 🐍 **MATLAB:** `ode45` ≈ `solve_ivp(method='RK45')`; `ode23` ≈ `method='RK23'`;
> `ode15s` ≈ `method='BDF'` (rígidas).

**Regla práctica:** Euler y Heun se enseñan para entender el concepto y son de
**bajo costo pero poca precisión**; en la práctica usa RK45 (o DOP853 para
alta precisión) y reserva BDF/LSODA para sistemas rígidos.

## 4. Preguntas teóricas

**P1.** El método de Euler tiene error local de truncamiento de orden…

1. $O(h)$
2. $O(h^2)$
3. $O(h^3)$
4. $O(h^4)$

**P2.** El método de Heun (Euler mejorado) es un método…

1. predictor-corrector que promedia dos pendientes
2. de pasos múltiples de cuarto orden
3. implícito para sistemas rígidos
4. de extrapolación de Richardson

In [ ]:
RESPUESTA_P1 = 2   # Euler: error local O(h^2) -> global O(h)
RESPUESTA_P2 = 1   # Heun = predictor-corrector

## 5. Caso de ingeniería: paracaidista en caída

La velocidad de un paracaidista de masa $m = 68.1$ kg con coeficiente de
arrastre $c = 12.5$ kg/s se modela con $\frac{dv}{dt} = g - \frac{c}{m}v$,
$v(0)=0$ (con $g = 9.81$ m/s²). Usa `solve_ivp` para simular los primeros
20 s y estima la **velocidad terminal** (cuando $dv/dt \to 0$).

In [ ]:
g, m, c = 9.81, 68.1, 12.5

def dvdt(t, v):
    return g - (c/m)*v

sol = solve_ivp(dvdt, (0, 20), [0.0], method='RK45', dense_output=True)
t = np.linspace(0, 20, 100)
v = sol.sol(t)[0]

v_terminal = g*m/c   # analítico: dv/dt=0 -> v = gm/c
print(f'Velocidad terminal (analítica) = {v_terminal:.2f} m/s')
print(f'Velocidad a t=20 s (numérica)   = {v[-1]:.2f} m/s')

plt.figure(figsize=(7, 4))
plt.plot(t, v, 'b-'); plt.axhline(v_terminal, color='r', ls='--', label='terminal')
plt.xlabel('t (s)'); plt.ylabel('v (m/s)'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Paracaidista en caída (RK45)'); plt.show()

RESPUESTA_CASO = float(v[-1])

## 6. Revisar, enviar y ver calificación

Las respuestas (`RESPUESTA_P1`, `RESPUESTA_P2`, `RESPUESTA_CASO`) se envían
al **Apps Script** (se conecta en la Fase 4).

In [ ]:
#@title 📤 Enviar respuestas y ver calificación
alumno_id = 'TU_NUMERO_CONTROL'
respuestas = {'P1': RESPUESTA_P1, 'P2': RESPUESTA_P2, 'CASO': RESPUESTA_CASO}
print('Pendiente de conectar con Apps Script:', respuestas)